# 01 · *Robustness* dan *Cross-Dataset Generalization* (Tahap B6–B10)
## Luaran artikel 2025: *Robustness and Cross-Dataset Generalization of Hybrid YOLO–CNN*

Notebook ini memakai hasil notebook 00 (data, *split* terkunci, label terverifikasi, model, `model_config.json`) dan menjalankan:

| Tahap | Pekerjaan | Keluaran utama |
|---|---|---|
| B6 | enam gangguan × tiga level (18 kondisi + *baseline*) | `perturbation_results.csv`, Tabel 6–7, Gambar 6–7 |
| B7 | uji eksternal dengan model beku (Roboflow `ivan-sthzz/dermatitis-zedaz`) | `external_results.csv`, audit *overlap*, Tabel 4 dan 8, Gambar 4 dan 8 |
| B8 | E1–E4, ΔL, ΔC, ΔP, ΔLP, rentang IoU | `error_propagation.csv`, Tabel 9–11, Gambar 9–10 |
| B9 | 95% CI *cluster bootstrap*, ECE, *Brier score*, metrik per kelas | `statistics.csv`, Tabel 12, Gambar 11 |
| B10 | ekspor tabel dan gambar | `PFR_2025_ROBUSTNESS/` dan ZIP |

### Empat kondisi propagasi kesalahan

Definisi mengikuti draf Subbab 3.5.1 (Hasil dan Pembahasan). Dokumen urutan pekerjaan menukar label E3/E4; notebook ini memakai penamaan di draf naskah:

| Kondisi | Citra | ROI masukan CNN |
|---|---|---|
| E1 | asli | ROI referensi (anotasi) |
| E2 | asli | ROI prediksi YOLO |
| E3 | terganggu | ROI referensi (koordinat sama, citra terganggu) |
| E4 | terganggu | ROI prediksi YOLO pada citra terganggu |

Dengan M = *macro-F1* (atau *balanced accuracy*):

- ΔL = M(E1) − M(E2): penalti ROI prediksi pada citra asli;
- ΔC = M(E1) − M(E3): degradasi klasifikasi saat lokasi ROI dikendalikan;
- ΔP = M(E2) − M(E4): degradasi *pipeline* akibat gangguan;
- ΔLP = [M(E3) − M(E4)] − [M(E1) − M(E2)] = ΔP − ΔC: perubahan penalti lokalisasi antara kondisi terganggu dan asli.

Cocokkan definisi ini dengan Persamaan (6)–(9) pada metode naskah. Karena *macro-F1* adalah metrik agregat nonlinier, dekomposisi ini bersifat deskriptif dan tidak mengidentifikasi efek kausal yang terisolasi.

## 0. Konfigurasi

- `MODEL_SETS_DIPAKAI`: model latih ulang (`retrain`, analisis utama) dan bobot lama (`lama`, pembanding *in-sample* pada data internal). Bobot lama dilewati otomatis bila tidak tersedia.
- Data eksternal: `RF_WORKSPACE`, `RF_PROJECT`, `RF_VERSION` (None = versi terbaru). API key Roboflow dibaca dari **Colab Secrets** dengan nama `ROBOFLOW_API_KEY` (ikon kunci di panel kiri Colab). Bila data eksternal sudah diunduh manual dalam format YOLO, isi `EKSTERNAL_LOKAL`.
- `EXT_LABEL_MAP = "auto"` memetakan nama kelas eksternal ke Ringan/Sedang/Berat berdasarkan kata kunci (mild/ringan, moderate/sedang, severe/berat). Isi dict `{id_kelas_eksternal: indeks_keparahan}` bila nama kelas berbeda.
- `B_BOOT` replikasi untuk metrik klasifikasi dan kalibrasi; `B_BOOT_DET` untuk mAP (lebih mahal).

In [ ]:
# ======================= CONFIG =======================
NAMA_PROYEK       = "YOLO_DERMA_PFR"
MODE_UJI_CEPAT    = False
MODEL_SETS_DIPAKAI = ("retrain", "lama")

# --- Data eksternal (B7) ---
RF_WORKSPACE      = "ivan-sthzz"
RF_PROJECT        = "dermatitis-zedaz"
RF_VERSION        = None        # None = versi terbaru
RF_FORMAT         = "yolov8"
EKSTERNAL_LOKAL   = None        # path dataset eksternal format YOLO (melewati unduhan Roboflow)
EXT_LABEL_MAP     = "auto"      # "auto" atau dict {id_kelas_eksternal: indeks_keparahan}
EXT_SPLITS        = ("train", "valid", "test")
EXT_JALANKAN_PERTURBASI = True

# --- Perturbasi dan statistik ---
PERTURB_SPEC      = None        # None = 6 gangguan x 3 level bawaan
TEST_MAKS_VARIAN_PER_SUMBER = None   # None = semua varian augmentasi di test set
B_BOOT            = 1000
B_BOOT_DET        = 300
N_BIN_ECE         = 15
IOU_BINS          = [0.0, 0.25, 0.50, 0.75, 1.0001]

## 1. Lingkungan, Drive, data, dan model

Sel ini memasang paket yang belum ada, me-*mount* Drive, membaca `config_run.json` dari notebook 00, memulihkan salinan repo dari *cache* Drive, memuat *split* terkunci, label terverifikasi, dan model sesuai `model_config.json`. PyTorch diimpor sebelum TensorFlow untuk menghindari konflik pustaka bersama.

In [ ]:
import os, sys, subprocess, importlib, time, json, shutil, glob, re, math, warnings, tarfile, datetime
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
IN_COLAB = "google.colab" in sys.modules
for mod, pkg in [("ultralytics", "ultralytics"), ("imagehash", "imagehash"), ("roboflow", "roboflow")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import torch, torchvision, ultralytics  # noqa: F401

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE, WORK = "/content/drive/MyDrive", "/content/work"
else:
    DRIVE_BASE, WORK = os.path.abspath("drive_lokal"), os.path.abspath("work_lokal")
if MODE_UJI_CEPAT:
    NAMA_PROYEK += "_UJI_CEPAT"
    B_BOOT, B_BOOT_DET = 100, 50
ROOT = os.path.join(DRIVE_BASE, NAMA_PROYEK)
P = {"kode": f"{ROOT}/00_kode", "data": f"{ROOT}/01_data_cache", "A": f"{ROOT}/A_PERSIAPAN",
     "A_tab": f"{ROOT}/A_PERSIAPAN/tables", "B": f"{ROOT}/PFR_2025_ROBUSTNESS"}
P.update({"B_tab": f"{P['B']}/tables", "B_fig": f"{P['B']}/figures", "B_raw": f"{P['B']}/raw", "B_log": f"{P['B']}/logs"})
for v in P.values():
    os.makedirs(v, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
cfg_path = os.path.join(P["A"], "config_run.json")
if not os.path.exists(cfg_path):
    raise FileNotFoundError("config_run.json tidak ditemukan. Jalankan notebook 00 terlebih dahulu.")
sys.path.insert(0, P["kode"])
import dermakit as dk
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
CFG = dk.core.load_json(cfg_path)
MC = dk.core.load_json(os.path.join(P["A_tab"], "model_config.json"))
SEED = CFG["SEED"]; dk.core.set_seed(SEED); dk.viz.setup_style()
LABEL_MAP = {int(k): v for k, v in CFG["LABEL_MAP"].items()}
STATUS_LABEL_MAP = CFG["STATUS_LABEL_MAP"]
CONF_THR, IOU_VALID = MC["inferensi"]["conf_thr"], MC["inferensi"]["iou_valid"]
NAMA_KELAS = dk.core.class_names(CFG.get("BAHASA_GAMBAR", "en")); NAMA_KELAS_ID = dk.core.class_names("id")
LOG = dk.core.RunLog(os.path.join(P["B_log"], "run_log_01.txt"))

REPO_DIR = CFG.get("REPO_LOKAL") or os.path.join(WORK, "YOLO-DERMA")
if not os.path.exists(os.path.join(REPO_DIR, "train")):
    with tarfile.open(os.path.join(P["data"], "yolo_derma_repo_cache.tar")) as t:
        t.extractall(WORK)
split_df = pd.read_csv(os.path.join(P["A_tab"], "locked_data_split.csv"))
ver = pd.read_csv(os.path.join(P["A_tab"], "verified_labels.csv"))
lock = dk.core.load_json(os.path.join(P["A_tab"], "split_lock.json"))
print("Repo:", REPO_DIR, "| split SHA-256:", lock["sha256_split"][:16], "| label map:", STATUS_LABEL_MAP)

tf = dk.models.get_tf()
import keras
def muat_model_sets():
    out = {}
    r = MC["retrain"]
    if "retrain" in MODEL_SETS_DIPAKAI and r.get("yolo_path") and os.path.exists(r["yolo_path"]):
        out["retrain"] = (dk.models.YoloRunner(r["yolo_path"], imgsz=MC["inferensi"]["yolo_imgsz"], nms_iou=MC["inferensi"]["nms_iou"],
                                               max_det=MC["inferensi"]["max_det"], name="retrain"),
                          dk.models.CnnRunner(keras.models.load_model(r["cnn_path"], compile=False), scale=r["cnn_scale"],
                                              class_order=r["cnn_class_order"], pad=r["cnn_pad"], name="retrain"))
    o = MC.get("lama", {})
    if "lama" in MODEL_SETS_DIPAKAI and o.get("tersedia"):
        out["lama"] = (dk.models.YoloRunner(o["yolo_path"], imgsz=MC["inferensi"]["yolo_imgsz"], nms_iou=MC["inferensi"]["nms_iou"],
                                            max_det=MC["inferensi"]["max_det"], keep_classes=o.get("yolo_keep_classes"), name="lama"),
                       dk.models.CnnRunner(dk.models.load_keras_any(o["cnn_path"])[0], scale=o["cnn_scale"],
                                           class_order=o["cnn_class_order"], pad=o["cnn_pad"], name="lama"))
    return out
MODEL_SETS = muat_model_sets()
print("Model dimuat:", list(MODEL_SETS))

## B6. Pengujian enam gangguan kualitas citra

### B6.1 Definisi kondisi dan contoh visual

Gangguan diterapkan pada citra utuh sebelum YOLO dengan bobot dan konfigurasi inferensi yang sama untuk semua kondisi. *Seed* *Gaussian noise* ditentukan oleh `sample_id`, sehingga citra yang sama selalu menerima pola *noise* yang sama dan perbandingan antar-kondisi bersifat berpasangan. Gambar contoh menampilkan satu citra *test* pada 19 kondisi; ciri klinis (eritema, papul, skuama) perlu diperiksa secara visual untuk memastikan level gangguan masih wajar secara klinis.

In [ ]:
SPEC = PERTURB_SPEC or dk.perturb.PERTURB_SPEC_DEFAULT
CONDS = dk.perturb.list_conditions(SPEC)
IDENT = {"brightness": 1.0, "contrast": 1.0, "gaussian_noise": 0.0, "gaussian_blur": 0.0, "resolution": 1.0, "jpeg": 100}
cond_df = pd.DataFrame(CONDS)
cond_df["param"] = cond_df["ptype"].map(lambda t: SPEC.get(t, {}).get("param", "-"))
cond_df.to_csv(os.path.join(P["B_tab"], "definisi_kondisi.csv"), index=False)
display(cond_df)

items_test = dk.pipeline.build_items(split_df, ver, REPO_DIR, "test")
if TEST_MAKS_VARIAN_PER_SUMBER:
    src_of = dict(zip(split_df.sample_id, split_df.base_source))
    cnt = {}
    sel = []
    for it in items_test:
        s = src_of[it["sample_id"]]
        if cnt.get(s, 0) < TEST_MAKS_VARIAN_PER_SUMBER:
            sel.append(it); cnt[s] = cnt.get(s, 0) + 1
    items_test = sel
print("Citra test:", len(items_test), "| lesi:", sum(len(i["gt_cls"]) for i in items_test),
      "| kelompok:", len({i["group"] for i in items_test}))

contoh = max(items_test, key=lambda i: len(set(i["gt_cls"].tolist())) * 100 + len(i["gt_cls"]))
img0 = dk.data.read_rgb(contoh["path"])
fig, ax = plt.subplots(4, 5, figsize=(16, 13.5))
ax = ax.ravel()
for k, c in enumerate(CONDS):
    im = dk.perturb.apply_perturbation(img0, c["ptype"], c["value"], seed_key=contoh["sample_id"])
    ax[k].imshow(im); ax[k].axis("off")
    ax[k].set_title("Original" if c["ptype"] == "original" else f"{c['label_en']}\n{c['value']}", fontsize=12)
for k in range(len(CONDS), len(ax)):
    ax[k].axis("off")
dk.viz.show_and_save(fig, os.path.join(P["B_fig"], "B6_contoh_perturbasi"))

### B6.2 Inferensi semua kondisi (dengan *cache* per kondisi)

Untuk setiap model dan setiap kondisi, `run_condition` menyimpan hasil mentah ke `PFR_2025_ROBUSTNESS/raw/<model>/<dataset>/`. Kondisi yang sudah tersimpan dilewati, sehingga sel ini dapat dijalankan ulang setelah sesi Colab terputus tanpa mengulang dari awal. Urutan baris ROI identik di semua kondisi (diperiksa dengan *assert*), syarat untuk *bootstrap* berpasangan.

In [ ]:
def raw_path(ms, ds, cid):
    return os.path.join(P["B_raw"], ms, ds, cid)

def jalankan_semua(items, ds_name, conds):
    for ms, (yr, cr) in MODEL_SETS.items():
        os.makedirs(os.path.join(P["B_raw"], ms, ds_name), exist_ok=True)
        for c in conds:
            base = raw_path(ms, ds_name, c["cond_id"])
            if dk.core.table_exists(base + "_roi") and os.path.exists(base + "_det.npz"):
                continue
            t0 = time.time()
            roi, img, det = dk.pipeline.run_condition(items, yr, cr, c, CONF_THR, IOU_VALID, desc=f"{ms}/{ds_name}/{c['cond_id']}")
            dk.core.save_table(roi, base + "_roi"); dk.core.save_table(img, base + "_img")
            dk.pipeline.save_det(det, base + "_det.npz")
            LOG(f"{ms}/{ds_name}/{c['cond_id']}: {len(roi)} ROI, {time.time() - t0:.0f} detik")

def muat_semua(ms, ds_name, conds):
    out = {}
    for c in conds:
        base = raw_path(ms, ds_name, c["cond_id"])
        if not dk.core.table_exists(base + "_roi"):
            continue
        roi = dk.core.load_table(base + "_roi"); img = dk.core.load_table(base + "_img")
        out[c["cond_id"]] = (roi, img, dk.pipeline.load_det(base + "_det.npz"))
    keys = list(out)
    if keys:
        ref = out[keys[0]][0][["sample_id", "roi_idx"]].to_numpy()
        for k in keys[1:]:
            assert (out[k][0][["sample_id", "roi_idx"]].to_numpy() == ref).all(), f"Urutan ROI berbeda pada {k}"
    return out

t0 = time.time()
jalankan_semua(items_test, "internal", CONDS)
print(f"Selesai dalam {(time.time() - t0) / 60:.1f} menit")
RAW = {ms: {"internal": muat_semua(ms, "internal", CONDS)} for ms in MODEL_SETS}

### B6.3 Metrik per kondisi dengan *bootstrap* klaster berpasangan

Untuk setiap model dan dataset dihitung, per kondisi:

- **Deteksi**: mAP@0.5, mAP@0.5:0.95, *precision* dan *recall* lesi pada `CONF_THR`, *detection failure rate* lesi (lesi tanpa ROI valid) dan citra (citra berlesi tanpa deteksi);
- **Klasifikasi**: *macro-F1*, *balanced accuracy*, dan *accuracy* untuk ROI referensi (E1 pada *original*, E3 pada kondisi terganggu) dan ROI prediksi (E2/E4);
- **Selisih terhadap *baseline***: Δ = *baseline* − kondisi (positif = penurunan) dan degradasi relatif (%) = Δ / *baseline* × 100.

Pada setiap replikasi *bootstrap*, kelompok yang sama dipakai untuk semua kondisi. Nilai p dua sisi bootstrap untuk H0: Δ = 0 dikoreksi Holm di antara 18 kondisi.

In [ ]:
def siapkan(raw):
    conds = list(raw)
    roi0 = raw[conds[0]][0]
    img0 = raw[conds[0]][1]
    img0 = img0[img0["read_ok"]] if "read_ok" in img0 else img0
    d = {"conds": conds, "true": roi0["true"].to_numpy(int), "roi_group": roi0["group"].astype(str).to_numpy(),
         "img_group": img0["group"].astype(str).to_numpy(), "sids": list(img0["sample_id"])}
    for c in conds:
        roi, img, det = raw[c]
        img = img[img["read_ok"]] if "read_ok" in img else img
        d[c] = {"ref": roi["ref_pred"].to_numpy(int), "pipe": roi["pipe_pred"].to_numpy(int),
                "valid": roi["valid_roi"].to_numpy(bool),
                "ref_p": roi[["ref_p0", "ref_p1", "ref_p2"]].to_numpy(float),
                "any_p": roi[["any_p0", "any_p1", "any_p2"]].to_numpy(float),
                "n_pred": img["n_pred"].to_numpy(float), "any_det": img["any_det"].to_numpy(bool),
                "img_has_gt": (img["n_gt"] > 0).to_numpy(bool),
                "det_flags": [det[s][0] for s in d["sids"]], "det_conf": [det[s][1] for s in d["sids"]],
                "det_ngt": np.array([det[s][2] for s in d["sids"]])}
    return d

def ap_idx(dc, idx):
    if len(idx) == 0:
        return np.nan, np.nan
    fl = np.concatenate([dc["det_flags"][i] for i in idx]); cf = np.concatenate([dc["det_conf"][i] for i in idx])
    ngt = int(dc["det_ngt"][idx].sum())
    if ngt == 0 or len(cf) == 0:
        return np.nan, np.nan
    o = np.argsort(-cf, kind="stable"); f = fl[o]
    tpc = np.cumsum(f, 0); fpc = np.cumsum(~f, 0)
    rec = tpc / ngt; prec = tpc / np.maximum(tpc + fpc, 1)
    aps = [dk.metrics.compute_ap(rec[:, t], prec[:, t]) for t in range(f.shape[1])]
    return aps[0], float(np.mean(aps))

def metrik_kondisi(d, c, ri, ii, with_det=True):
    dc, yt = d[c], d["true"][ri]
    out = {"cls_ref_macro_f1": dk.metrics.macro_f1_fast(yt, dc["ref"][ri]),
           "cls_pred_macro_f1": dk.metrics.macro_f1_fast(yt, dc["pipe"][ri]),
           "cls_ref_bal_acc": dk.metrics.balanced_acc_fast(yt, dc["ref"][ri]),
           "cls_pred_bal_acc": dk.metrics.balanced_acc_fast(yt, dc["pipe"][ri]),
           "cls_ref_accuracy": float(np.mean(yt == dc["ref"][ri])) if len(ri) else np.nan,
           "cls_pred_accuracy": float(np.mean(yt == dc["pipe"][ri])) if len(ri) else np.nan,
           "det_failure_rate_lesi": float(np.mean(~dc["valid"][ri])) if len(ri) else np.nan,
           "det_recall_thr": float(np.mean(dc["valid"][ri])) if len(ri) else np.nan}
    npred = dc["n_pred"][ii].sum()
    out["det_precision_thr"] = float(dc["valid"][ri].sum() / npred) if npred else np.nan
    hg = dc["img_has_gt"][ii]
    out["det_failure_rate_citra"] = float(np.mean(~dc["any_det"][ii][hg])) if hg.any() else np.nan
    if with_det:
        out["det_map50"], out["det_map50_95"] = ap_idx(dc, ii)
    return out

def evaluasi(raw, B=B_BOOT, B_det=B_BOOT_DET, seed=SEED):
    d = siapkan(raw)
    conds = d["conds"]
    all_r, all_i = np.arange(len(d["true"])), np.arange(len(d["sids"]))
    est = {c: metrik_kondisi(d, c, all_r, all_i) for c in conds}
    cb = dk.stats.ClusterBootstrap(B, seed).register("roi", d["roi_group"]).register("img", d["img_group"])
    boot = {c: [] for c in conds}
    for b, dr in enumerate(cb.draws()):
        for c in conds:
            boot[c].append(metrik_kondisi(d, c, dr["roi"], dr["img"], with_det=b < B_det))
    boot = {c: pd.DataFrame(v) for c, v in boot.items()}
    return d, est, boot

HIGHER_BETTER = {"det_failure_rate_lesi": False, "det_failure_rate_citra": False}
def ringkas_kondisi(est, boot, ms, ds):
    rows = []
    base = "original"
    metr = list(est[base].keys())
    for c in est:
        info = next(x for x in CONDS if x["cond_id"] == c)
        for m in metr:
            e = est[c][m]; lo, hi = dk.stats.percentile_ci(boot[c][m])
            row = {"model": ms, "dataset": ds, "cond_id": c, "ptype": info["ptype"], "level": info["level"], "value": info["value"],
                   "metric": m, "estimate": e, "ci_lo": lo, "ci_hi": hi}
            if c != base:
                dlt = boot[base][m].to_numpy() - boot[c][m].to_numpy()
                row["delta"] = est[base][m] - e
                row["delta_ci_lo"], row["delta_ci_hi"] = dk.stats.percentile_ci(dlt)
                row["rel_degradation_pct"] = 100 * (est[base][m] - e) / est[base][m] if est[base][m] else np.nan
                row["p_boot"] = dk.stats.boot_p_two_sided(dlt)
            rows.append(row)
    df = pd.DataFrame(rows)
    df["p_holm"] = np.nan
    for m in metr:
        msk = (df.metric == m) & (df.cond_id != base)
        df.loc[msk, "p_holm"] = dk.stats.holm(df.loc[msk, "p_boot"].to_numpy())
    return df

EVAL = {}
t0 = time.time()
for ms in MODEL_SETS:
    d, est, boot = evaluasi(RAW[ms]["internal"])
    EVAL[(ms, "internal")] = (d, est, boot)
    print(f"{ms}/internal selesai ({(time.time() - t0) / 60:.1f} menit)")
pert_res = pd.concat([ringkas_kondisi(EVAL[k][1], EVAL[k][2], *k) for k in EVAL], ignore_index=True)
pert_res["label_map_status"] = STATUS_LABEL_MAP
pert_res.to_csv(os.path.join(P["B_tab"], "perturbation_results.csv"), index=False)
print("Tersimpan: perturbation_results.csv", pert_res.shape)

### B6.4 Deteksi YOLO di bawah gangguan (Tabel 6, Gambar 6)

Tabel 6 menguraikan setiap jenis gangguan menjadi tiga baris level. *Detection failure rate* lesi = proporsi lesi referensi tanpa kotak prediksi berpasangan dengan IoU ≥ 0,5 pada `CONF_THR`. Gambar 6 menempatkan *baseline* pada nilai identitas tiap gangguan (*brightness/contrast* = 1, σ = 0, resolusi = 100%, Q = 100) sehingga arah dan besar perubahan terbaca langsung.

In [ ]:
def tabel_pivot(df, ms, ds, metrics, nama):
    rows = []
    sub = df[(df.model == ms) & (df.dataset == ds)]
    for c in [x["cond_id"] for x in CONDS]:
        s = sub[sub.cond_id == c]
        if s.empty:
            continue
        info = next(x for x in CONDS if x["cond_id"] == c)
        row = {"Gangguan": "Original" if info["ptype"] == "original" else info["label_en"],
               "Level": "Baseline" if info["level"] == 0 else f"L{info['level']} ({info['value']})"}
        for m, lab in zip(metrics, nama):
            r = s[s.metric == m].iloc[0]
            row[lab] = f"{r.estimate:.3f} [{r.ci_lo:.3f}; {r.ci_hi:.3f}]"
        rows.append(row)
    return pd.DataFrame(rows)

tab6 = tabel_pivot(pert_res, "retrain", "internal", ["det_map50", "det_map50_95", "det_failure_rate_lesi", "det_failure_rate_citra"],
                   ["mAP@0.5 (95% CI)", "mAP@0.5:0.95 (95% CI)", "Detection failure rate lesi (95% CI)", "Failure rate citra (95% CI)"])
tab6.to_csv(os.path.join(P["B_tab"], "tabel6_deteksi_perturbasi.csv"), index=False)
display(tab6)

def plot_kurva(df, ms, ds, metrics, labels, fname, ylabel_rows):
    ptypes = [p for p in dk.viz.PTYPE_ORDER if p in SPEC]
    fig, ax = plt.subplots(len(ylabel_rows), len(ptypes), figsize=(3.2 * len(ptypes), 3.4 * len(ylabel_rows)), sharey="row", squeeze=False)
    sub = df[(df.model == ms) & (df.dataset == ds)]
    for r_i, row_metrics in enumerate(metrics):
        for c_i, pt in enumerate(ptypes):
            a = ax[r_i, c_i]
            palet = [["#2F6690", "#D1495B"], ["#6A4C93", "#E0A030"]][r_i % 2]
            for m, lab, col in zip(row_metrics, labels[r_i], palet):
                b = sub[(sub.metric == m) & (sub.cond_id == "original")].iloc[0]
                s = sub[(sub.metric == m) & (sub.ptype == pt)]
                xs = [IDENT[pt]] + list(s.value.astype(float)); ys = [b.estimate] + list(s.estimate)
                lo = [b.ci_lo] + list(s.ci_lo); hi = [b.ci_hi] + list(s.ci_hi)
                o = np.argsort(xs)
                xs, ys, lo, hi = np.array(xs)[o], np.array(ys)[o], np.array(lo)[o], np.array(hi)[o]
                a.plot(xs, ys, "-o", color=col, label=lab, ms=5)
                a.fill_between(xs, lo, hi, color=col, alpha=0.15)
            a.axvline(IDENT[pt], color="gray", ls=":", lw=1)
            if r_i == 0:
                a.set_title(SPEC[pt].get("label_en", pt))
            if r_i == len(metrics) - 1:
                a.set_xlabel(SPEC[pt].get("param_en", SPEC[pt]["param"]))
            if c_i == 0:
                a.set_ylabel(ylabel_rows[r_i])
                a.legend(loc="best", fontsize=11, framealpha=0.85)
            if pt in ("jpeg", "resolution"):
                a.invert_xaxis()
    return dk.viz.show_and_save(fig, os.path.join(P["B_fig"], fname))

plot_kurva(pert_res, "retrain", "internal", [["det_map50", "det_map50_95"], ["det_failure_rate_lesi", "det_failure_rate_citra"]],
           [["mAP@0.5", "mAP@0.5:0.95"], ["Lesion failure rate", "Image failure rate"]],
           "Gambar6_robustness_deteksi_YOLO", ["mAP", "Failure rate"])

In [ ]:
fmt, fci, pct = dk.teks.fmt, dk.teks.fmt_ci, dk.teks.pct
def interpretasi_metrik(df, ms, ds, metric, nama, lebih_tinggi_baik=True):
    sub = df[(df.model == ms) & (df.dataset == ds) & (df.metric == metric)]
    b = sub[sub.cond_id == "original"].iloc[0]
    s = sub[sub.cond_id != "original"].copy()
    s["turun"] = s["delta"] if lebih_tinggi_baik else -s["delta"]
    w = s.loc[s["turun"].idxmax()]
    bermakna = s[(s.p_holm < 0.05) & (s["turun"] > 0)]
    arah_baik = "lebih tinggi lebih baik" if lebih_tinggi_baik else "lebih rendah lebih baik"
    teks = [f"{nama} baseline = {fmt(b.estimate)} (95% CI {fci(b.ci_lo, b.ci_hi)}; {arah_baik})."]
    teks.append(f"Perburukan terbesar terjadi pada {w.cond_id} (nilai {float(w.value):g}): {nama} menjadi {fmt(w.estimate)}, "
                f"besar perburukan {fmt(w.turun)} (95% CI {fci(*(sorted([v if lebih_tinggi_baik else -v for v in (w.delta_ci_lo, w.delta_ci_hi)])))}).")
    teks.append(f"{len(bermakna)} dari {len(s)} kondisi menunjukkan perburukan dengan p Holm < 0,05"
                + (": " + ", ".join(bermakna.sort_values('turun', ascending=False).cond_id) + "." if len(bermakna) else "."))
    per_tipe = s.groupby("ptype")["turun"].max().sort_values(ascending=False)
    teks.append("Urutan jenis gangguan menurut perburukan maksimum (nilai negatif = membaik): " + ", ".join(f"{k} ({fmt(v)})" for k, v in per_tipe.items()) + ".")
    return " ".join(teks)

print(interpretasi_metrik(pert_res, "retrain", "internal", "det_map50", "mAP@0.5"))
print()
print(interpretasi_metrik(pert_res, "retrain", "internal", "det_map50_95", "mAP@0.5:0.95"))
print()
print(interpretasi_metrik(pert_res, "retrain", "internal", "det_failure_rate_lesi", "Detection failure rate lesi", lebih_tinggi_baik=False))

### B6.5 Klasifikasi CNN dan *end-to-end* di bawah gangguan (Tabel 7, Gambar 7)

Kolom *macro-F1* ROI referensi memakai E3 (dan E1 untuk *baseline*); kolom ROI prediksi memakai E4 (dan E2 untuk *baseline*). Degradasi relatif dihitung terhadap *baseline* masing-masing. Kesenjangan antara kedua kurva menunjukkan apakah gangguan tertentu memperbesar penalti penggunaan ROI hasil YOLO.

In [ ]:
sub = pert_res[(pert_res.model == "retrain") & (pert_res.dataset == "internal")]
rows = []
for c in [x["cond_id"] for x in CONDS]:
    info = next(x for x in CONDS if x["cond_id"] == c)
    r1 = sub[(sub.cond_id == c) & (sub.metric == "cls_ref_macro_f1")].iloc[0]
    r2 = sub[(sub.cond_id == c) & (sub.metric == "cls_pred_macro_f1")].iloc[0]
    rows.append({"Gangguan": "Original" if info["ptype"] == "original" else info["label_en"],
                 "Level": "Baseline" if info["level"] == 0 else f"L{info['level']} ({info['value']})",
                 "Macro-F1 ROI referensi (95% CI)": f"{r1.estimate:.3f} [{r1.ci_lo:.3f}; {r1.ci_hi:.3f}]",
                 "Macro-F1 ROI prediksi (95% CI)": f"{r2.estimate:.3f} [{r2.ci_lo:.3f}; {r2.ci_hi:.3f}]",
                 "Degradasi relatif ROI referensi (%)": 0.0 if c == "original" else round(r1.rel_degradation_pct, 2),
                 "Degradasi relatif ROI prediksi (%)": 0.0 if c == "original" else round(r2.rel_degradation_pct, 2),
                 "p Holm (ROI prediksi)": np.nan if c == "original" else round(r2.p_holm, 4)})
tab7 = pd.DataFrame(rows)
tab7.to_csv(os.path.join(P["B_tab"], "tabel7_klasifikasi_perturbasi.csv"), index=False)
display(tab7)
plot_kurva(pert_res, "retrain", "internal", [["cls_ref_macro_f1", "cls_pred_macro_f1"], ["cls_ref_bal_acc", "cls_pred_bal_acc"]],
           [["Reference ROI", "YOLO-predicted ROI"], ["Reference ROI", "YOLO-predicted ROI"]],
           "Gambar7_degradasi_macroF1", ["Macro-F1", "Balanced accuracy"])
print(interpretasi_metrik(pert_res, "retrain", "internal", "cls_ref_macro_f1", "Macro-F1 ROI referensi"))
print()
print(interpretasi_metrik(pert_res, "retrain", "internal", "cls_pred_macro_f1", "Macro-F1 ROI prediksi (end-to-end)"))
LOG("B6 selesai.")

## B7. Uji eksternal dengan model beku

### B7.1 Unduh dan inspeksi data eksternal

Dataset eksternal diunduh dari Roboflow Universe (`RF_WORKSPACE/RF_PROJECT`) dalam format YOLO lalu disimpan sebagai *cache* tar di Drive. Model tidak dilatih ulang, bobot tidak diperbarui, dan ambang tidak disetel memakai label eksternal. Semua *split* eksternal (*train/valid/test*) dipakai sebagai data uji karena model tidak pernah melihatnya.

Pemetaan label eksternal:

1. `EXT_LABEL_MAP = "auto"`: nama kelas yang mengandung *mild/ringan/light* → Ringan, *moderate/sedang/medium* → Sedang, *severe/berat/heavy* → Berat. Kelas lain (misalnya *normal*, *healthy*, nama proyek) dikeluarkan.
2. Bila tidak ada kelas keparahan yang dapat dipetakan, data eksternal hanya dipakai untuk uji lokalisasi lesi; metrik klasifikasi eksternal dilaporkan NA karena tidak ada standar referensi keparahan yang setara.

Lisensi dataset Roboflow Universe perlu dicantumkan dan disitasi di naskah sesuai halaman dataset.

In [ ]:
import yaml
EXT_ROOT = os.path.join(WORK, "eksternal")
ext_tar = os.path.join(P["data"], f"eksternal_{RF_WORKSPACE}_{RF_PROJECT}_v{RF_VERSION or 'latest'}.tar")
EXT_DIR, ext_meta = None, {}
if EKSTERNAL_LOKAL:
    EXT_DIR = EKSTERNAL_LOKAL
elif os.path.exists(ext_tar):
    with tarfile.open(ext_tar) as t:
        t.extractall(EXT_ROOT)
    EXT_DIR = os.path.join(EXT_ROOT, os.listdir(EXT_ROOT)[0])
    ext_meta = dk.core.load_json(ext_tar.replace(".tar", ".json")) if os.path.exists(ext_tar.replace(".tar", ".json")) else {}
else:
    try:
        key = None
        if IN_COLAB:
            from google.colab import userdata
            try:
                key = userdata.get("ROBOFLOW_API_KEY")
            except Exception:
                key = None
        key = key or os.environ.get("ROBOFLOW_API_KEY")
        if not key:
            import getpass
            key = getpass.getpass("ROBOFLOW_API_KEY: ")
        from roboflow import Roboflow
        proj = Roboflow(api_key=key).workspace(RF_WORKSPACE).project(RF_PROJECT)
        vnum = RF_VERSION or max(int(str(v.version).split("/")[-1]) for v in proj.versions())
        loc = os.path.join(EXT_ROOT, f"{RF_PROJECT}-v{vnum}")
        proj.version(int(vnum)).download(RF_FORMAT, location=loc, overwrite=True)
        EXT_DIR = loc
        ext_meta = {"workspace": RF_WORKSPACE, "project": RF_PROJECT, "version": int(vnum), "format": RF_FORMAT,
                    "diunduh": datetime.datetime.now().isoformat(timespec="seconds"),
                    "url": f"https://universe.roboflow.com/{RF_WORKSPACE}/{RF_PROJECT}/dataset/{vnum}"}
        with tarfile.open(ext_tar, "w") as t:
            t.add(EXT_DIR, arcname=os.path.basename(EXT_DIR))
        dk.core.save_json(ext_meta, ext_tar.replace(".tar", ".json"))
    except Exception as e:
        print("Unduhan data eksternal gagal:", type(e).__name__, str(e)[:300])
        EXT_DIR = None
ADA_EKSTERNAL = EXT_DIR is not None and os.path.exists(EXT_DIR)
print("Data eksternal:", EXT_DIR if ADA_EKSTERNAL else "TIDAK TERSEDIA (B7 dilewati; klaim terbatas pada robustness internal)")
print(ext_meta)

In [ ]:
def peta_otomatis(names):
    kunci = {0: ["mild", "ringan", "light", "low"], 1: ["moderate", "sedang", "medium", "mid"], 2: ["severe", "berat", "heavy", "high"]}
    m = {}
    for i, n in names.items():
        nl = str(n).lower()
        for k, kws in kunci.items():
            if any(w in nl for w in kws):
                m[int(i)] = k
    return m

if ADA_EKSTERNAL:
    yml = [p for p in glob.glob(os.path.join(EXT_DIR, "*.yaml"))]
    ext_yaml = yaml.safe_load(open(yml[0])) if yml else {}
    names = ext_yaml.get("names", {})
    names = {i: n for i, n in enumerate(names)} if isinstance(names, list) else {int(k): v for k, v in names.items()}
    EXT_MAP = peta_otomatis(names) if EXT_LABEL_MAP == "auto" else {int(k): int(v) for k, v in EXT_LABEL_MAP.items()}
    EXT_ADA_KEPARAHAN = len(set(EXT_MAP.values())) >= 2
    if not EXT_MAP:
        EXT_MAP = {i: 0 for i, n in names.items() if not any(w in str(n).lower() for w in ["normal", "healthy", "sehat"])}
    print("Kelas eksternal:", names)
    print("Pemetaan ke keparahan:", {names.get(k, k): NAMA_KELAS_ID[v] for k, v in EXT_MAP.items()} if EXT_ADA_KEPARAHAN else EXT_MAP)
    print("Label keparahan setara tersedia:", EXT_ADA_KEPARAHAN)

    erows = []
    for sp in EXT_SPLITS:
        for p in dk.data.list_images(os.path.join(EXT_DIR, sp, "images")):
            rec = dk.data.image_record(p)
            stem = os.path.splitext(os.path.basename(p))[0]
            lp = os.path.join(EXT_DIR, sp, "labels", stem + ".txt")
            arr, bad = dk.data.read_yolo_label(lp)
            src = dk.data.parse_source(stem)
            erows.append({"path": p, "ext_split": sp, "stem": stem, "source_stem": src["source_stem"],
                          "base_source": dk.data.base_source(src["source_stem"]), "sha256": rec["sha256"],
                          "phash": rec["phash"], "dhash": rec["dhash"], "width": rec["width"], "height": rec["height"],
                          "readable": rec["readable"], "n_boxes": len(arr), "n_bad_rows": bad,
                          "label_ids": arr[:, 0].astype(int).tolist() if len(arr) else []})
    ext = pd.DataFrame(erows)
    ext["sample_id"] = "EXT_" + ext["sha256"].str[:12]
    r_ = ext.groupby("sample_id").cumcount()
    ext.loc[r_ > 0, "sample_id"] = ext.loc[r_ > 0, "sample_id"] + "_" + (r_[r_ > 0] + 1).astype(str)
    print("Citra eksternal:", len(ext), "| per split:", ext.ext_split.value_counts().to_dict())

### B7.2 Audit *overlap* eksternal–internal dan pengelompokan eksternal

Data eksternal harus independen: tidak ada citra yang sama atau nyaris sama dengan data internal (*train*, *val*, maupun *test*). Citra eksternal dikeluarkan bila:

- SHA-256 identik dengan citra internal;
- jarak pHash **dan** dHash ≤ ambang (aturan konjungtif yang sama dengan A1) terhadap citra internal mana pun;
- nama sumber (misalnya `Screenshot-<tanggal>-<jam>`) sama dengan sumber internal.

Salinan duplikat di dalam data eksternal juga dikeluarkan. Citra eksternal yang tersisa dikelompokkan per sumber Roboflow dan *near-duplicate* internal-eksternal (*union-find*), lalu kelompok ini menjadi unit *bootstrap*. Hasil audit: `external_overlap_audit.csv`.

In [ ]:
if ADA_EKSTERNAL:
    man = pd.read_csv(os.path.join(P["A_tab"], "dataset_manifest.csv"))
    ih_p = [int(h, 16) if isinstance(h, str) else None for h in man["phash"]]
    ih_d = [int(h, 16) if isinstance(h, str) else None for h in man["dhash"]]
    thr = CFG["HASH_HAMMING_THR"]
    hit = {}
    pp = {(i, j): d for i, j, d in dk.data.near_duplicate_pairs(ext["phash"].tolist(), ih_p, thr=thr)}
    dd_ = {(i, j): d for i, j, d in dk.data.near_duplicate_pairs(ext["dhash"].tolist(), ih_d, thr=thr)}
    for (i, j) in set(pp) & set(dd_):
        hit.setdefault(i, []).append(f"{man.iloc[j].sample_id}:p{pp[(i, j)]}:d{dd_[(i, j)]}")
    ext["overlap_hash"] = [";".join(hit.get(i, [])) for i in range(len(ext))]
    ext["overlap_sha"] = ext["sha256"].isin(set(man["sha256"]))
    ext["overlap_source"] = ext["base_source"].isin(set(man["base_source"]))
    ext["dup_internal_ext"] = ext.duplicated("sha256", keep="first")
    ext["dikeluarkan"] = (ext["overlap_hash"] != "") | ext["overlap_sha"] | ext["overlap_source"] | ext["dup_internal_ext"] | ~ext["readable"]
    ext["alasan"] = np.select([~ext["readable"], ext["overlap_sha"], ext["overlap_hash"] != "", ext["overlap_source"], ext["dup_internal_ext"]],
                              ["tidak terbaca", "SHA-256 sama dengan internal", "near-duplicate internal", "sumber sama dengan internal",
                               "duplikat di dalam eksternal"], default="")
    ext.drop(columns=["phash", "dhash"]).to_csv(os.path.join(P["B_tab"], "external_overlap_audit.csv"), index=False)
    ext_ok = ext[~ext["dikeluarkan"]].reset_index(drop=True)
    uf = dk.data.UnionFind(ext_ok["sample_id"].tolist())
    for bs, g in ext_ok.groupby("base_source"):
        ids = g["sample_id"].tolist()
        for s in ids[1:]:
            uf.union(ids[0], s)
    p_in = {(i, j) for i, j, _ in dk.data.near_duplicate_pairs(ext_ok["phash"].tolist(), thr=thr)}
    d_in = {(i, j) for i, j, _ in dk.data.near_duplicate_pairs(ext_ok["dhash"].tolist(), thr=thr)}
    for i, j in p_in & d_in:
        uf.union(ext_ok.sample_id[i], ext_ok.sample_id[j])
    rt = uf.groups(); gmap = {r: f"E{i:04d}" for i, r in enumerate(sorted(set(rt.values())))}
    ext_ok["group_id"] = ext_ok["sample_id"].map(lambda s: gmap[rt[s]])
    print(ext["alasan"].replace("", "lolos").value_counts().to_string())
    print(f"Citra eksternal lolos: {len(ext_ok)} dari {len(ext)}; kelompok: {ext_ok.group_id.nunique()}")

    items_ext = []
    for r in ext_ok.itertuples():
        arr, _ = dk.data.read_yolo_label(os.path.join(EXT_DIR, r.ext_split, "labels", r.stem + ".txt"))
        if len(arr):
            keep = np.array([int(c) in EXT_MAP for c in arr[:, 0]])
            arr = arr[keep]
        xyxy = dk.data.yolo_to_xyxy(arr[:, 1:5], r.width, r.height) if len(arr) else np.zeros((0, 4), np.float32)
        ok = (np.minimum(xyxy[:, 2] - xyxy[:, 0], xyxy[:, 3] - xyxy[:, 1]) >= CFG["MIN_SISI_KOTAK_PX"]) if len(xyxy) else np.zeros(0, bool)
        cls = np.array([EXT_MAP[int(c)] for c in arr[:, 0]], int) if len(arr) else np.zeros(0, int)
        items_ext.append({"sample_id": r.sample_id, "group": r.group_id, "path": r.path, "gt_xyxy": xyxy[ok], "gt_cls": cls[ok]})
    print("Lesi eksternal (setelah pemetaan):", sum(len(i["gt_cls"]) for i in items_ext))

### B7.3 Karakteristik dataset internal dan eksternal (Tabel 4, Gambar 4)

Selain distribusi kelas, kita membandingkan deskriptor domain sederhana per citra: resolusi asli, rerata L\* (kecerahan), dan rerata a\* (kemerahan) CIELAB. Perbedaan deskriptor ini membantu menafsirkan *generalization gap* tanpa langsung menganggapnya akibat satu faktor.

In [ ]:
import cv2
def deskriptor(paths, n=300, seed=SEED):
    rng = np.random.default_rng(seed)
    sel = [paths[i] for i in rng.permutation(len(paths))[:n]]
    out = []
    for p in sel:
        im = dk.data.read_rgb(p)
        if im is None:
            continue
        lab = cv2.cvtColor(im, cv2.COLOR_RGB2LAB).astype(np.float32)
        v = lab[..., 0] > 5
        out.append({"L_star": lab[..., 0][v].mean() * 100 / 255, "a_star": lab[..., 1][v].mean() - 128, "w": im.shape[1], "h": im.shape[0]})
    return pd.DataFrame(out)

tab4_int = pd.read_csv(os.path.join(P["A_tab"], "tabel4_internal.csv"))
tab4 = tab4_int.copy()
items_int_all = dk.pipeline.build_items(split_df, ver, REPO_DIR)
d_int = deskriptor([i["path"] for i in items_int_all])
if ADA_EKSTERNAL:
    lesi_ext = np.concatenate([i["gt_cls"] for i in items_ext]) if items_ext else np.zeros(0, int)
    sev_img_ext = [int(i["gt_cls"].max()) if len(i["gt_cls"]) else -1 for i in items_ext]
    ev = {"Total citra sebelum audit": len(ext), "Citra duplikat persis": int(ext["dup_internal_ext"].sum()),
          "Citra dengan label tidak valid/tidak ada": int((ext.n_bad_rows > 0).sum()), "Citra lolos audit": len(ext_ok),
          "Sumber screenshot unik (lolos audit)": ext_ok.base_source.nunique(), "Kelompok anti-leakage": ext_ok.group_id.nunique(),
          "Pasien unik": "NA",
          "Citra Ringan (maks)": sev_img_ext.count(0) if EXT_ADA_KEPARAHAN else "NA",
          "Citra Sedang (maks)": sev_img_ext.count(1) if EXT_ADA_KEPARAHAN else "NA",
          "Citra Berat (maks)": sev_img_ext.count(2) if EXT_ADA_KEPARAHAN else "NA",
          "Citra tanpa lesi terverifikasi": sev_img_ext.count(-1),
          "Lesi Ringan": int((lesi_ext == 0).sum()) if EXT_ADA_KEPARAHAN else "NA",
          "Lesi Sedang": int((lesi_ext == 1).sum()) if EXT_ADA_KEPARAHAN else "NA",
          "Lesi Berat": int((lesi_ext == 2).sum()) if EXT_ADA_KEPARAHAN else "NA",
          "Rentang resolusi": f"{int(ext_ok.width.min())}x{int(ext_ok.height.min())} - {int(ext_ok.width.max())}x{int(ext_ok.height.max())}",
          "Citra test (split terkunci)": len(ext_ok)}
    tab4["Dataset Eksternal"] = tab4["Karakteristik"].map(lambda k: ev.get(k, "NA"))
    tab4.loc[len(tab4)] = ["Citra dikeluarkan karena overlap dengan internal", 0,
                           int(((ext["overlap_hash"] != "") | ext["overlap_sha"] | ext["overlap_source"]).sum())]
    d_ext = deskriptor(ext_ok["path"].tolist())
else:
    tab4["Dataset Eksternal"] = "NA"
    d_ext = pd.DataFrame(columns=d_int.columns)
for nm, dd in [("Rerata L* (kecerahan) +/- SD", "L_star"), ("Rerata a* (kemerahan) +/- SD", "a_star")]:
    tab4.loc[len(tab4)] = [nm, f"{d_int[dd].mean():.2f} +/- {d_int[dd].std():.2f}",
                           f"{d_ext[dd].mean():.2f} +/- {d_ext[dd].std():.2f}" if len(d_ext) else "NA"]
tab4["label_map_status"] = STATUS_LABEL_MAP
tab4.to_csv(os.path.join(P["B_tab"], "tabel4_dataset.csv"), index=False)
display(tab4)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
lesi_int = np.concatenate([i["gt_cls"] for i in items_int_all])
sets = [("Internal", lesi_int, "#2F6690")] + ([("External", lesi_ext, "#D1495B")] if ADA_EKSTERNAL and EXT_ADA_KEPARAHAN else [])
w = 0.8 / len(sets)
for k, (nm, arr, col) in enumerate(sets):
    prop = np.bincount(arr, minlength=3)[:3] / max(len(arr), 1)
    bars = ax[0].bar(np.arange(3) + k * w - 0.4 + w / 2, prop, w, label=f"{nm} (n={len(arr):,})", color=col)
    ax[0].bar_label(bars, labels=[f"{v:.2f}" for v in prop], fontsize=11, padding=2)
ax[0].set_xticks(range(3)); ax[0].set_xticklabels(NAMA_KELAS); ax[0].set_ylabel("Proportion of lesions")
ax[0].set_title("(a) Lesion severity distribution"); ax[0].legend(loc="upper left")
for a, col_, judul in [(ax[1], "L_star", "(b) Mean L* per image"), (ax[2], "a_star", "(c) Mean a* per image")]:
    a.hist(d_int[col_], bins=30, alpha=0.6, density=True, label="Internal", color="#2F6690")
    if len(d_ext):
        a.hist(d_ext[col_], bins=30, alpha=0.6, density=True, label="External", color="#D1495B")
    a.set_title(judul); a.set_ylabel("Density"); a.legend()
dk.viz.show_and_save(fig, os.path.join(P["B_fig"], "Gambar4_karakteristik_dataset"))

### B7.4 Inferensi eksternal, Tabel 8, dan Gambar 8

Inferensi eksternal memakai fungsi, ambang, dan bobot yang sama dengan evaluasi internal. *Generalization gap* = internal − eksternal untuk metrik yang semakin tinggi semakin baik, dan eksternal − internal untuk *failure rate*. CI gap dihitung dari dua *bootstrap* klaster independen (internal dan eksternal tidak berpasangan).

In [ ]:
if ADA_EKSTERNAL:
    conds_ext = CONDS if EXT_JALANKAN_PERTURBASI else CONDS[:1]
    jalankan_semua(items_ext, "eksternal", conds_ext)
    for ms in MODEL_SETS:
        RAW[ms]["eksternal"] = muat_semua(ms, "eksternal", conds_ext)
        d, est, boot = evaluasi(RAW[ms]["eksternal"], seed=SEED + 1)
        EVAL[(ms, "eksternal")] = (d, est, boot)
    pert_ext = pd.concat([ringkas_kondisi(EVAL[(ms, "eksternal")][1], EVAL[(ms, "eksternal")][2], ms, "eksternal") for ms in MODEL_SETS],
                         ignore_index=True)
    pert_res = pd.concat([pert_res[pert_res.dataset != "eksternal"], pert_ext], ignore_index=True)
    pert_res["label_map_status"] = STATUS_LABEL_MAP
    pert_res.to_csv(os.path.join(P["B_tab"], "perturbation_results.csv"), index=False)

    metr8 = [("det_map50", "YOLO mAP@0.5", True), ("det_map50_95", "YOLO mAP@0.5:0.95", True),
             ("cls_ref_macro_f1", "CNN macro-F1 (ROI referensi)", True), ("cls_pred_macro_f1", "Pipeline macro-F1", True),
             ("cls_pred_bal_acc", "Pipeline balanced accuracy", True), ("det_failure_rate_lesi", "Detection failure rate", False)]
    LABEL_EN8 = {"CNN macro-F1 (ROI referensi)": "CNN macro-F1 (reference ROI)"}
    rows = []
    for ms in MODEL_SETS:
        _, ei, bi = EVAL[(ms, "internal")]; _, ee, be = EVAL[(ms, "eksternal")]
        for m, lab, hb in metr8:
            klas = m.startswith("cls_")
            if klas and not EXT_ADA_KEPARAHAN:
                rows.append({"model": ms, "Metrik": lab, "Internal": ei["original"][m], "Eksternal": np.nan, "Generalization gap": np.nan,
                             "ci_lo": np.nan, "ci_hi": np.nan, "catatan": "label keparahan eksternal tidak tersedia"})
                continue
            bi_m, be_m = bi["original"][m].to_numpy(), be["original"][m].to_numpy()
            n = min(np.isfinite(bi_m).sum(), np.isfinite(be_m).sum())
            gb = (bi_m[np.isfinite(bi_m)][:n] - be_m[np.isfinite(be_m)][:n]) if hb else (be_m[np.isfinite(be_m)][:n] - bi_m[np.isfinite(bi_m)][:n])
            gap = (ei["original"][m] - ee["original"][m]) if hb else (ee["original"][m] - ei["original"][m])
            lo, hi = dk.stats.percentile_ci(gb)
            rows.append({"model": ms, "Metrik": lab, "Internal": ei["original"][m], "Eksternal": ee["original"][m],
                         "Generalization gap": gap, "ci_lo": lo, "ci_hi": hi,
                         "catatan": "in-sample internal" if ms == "lama" else ""})
    tab8 = pd.DataFrame(rows)
    tab8["95% CI Gap"] = tab8.apply(lambda r: f"[{r.ci_lo:.3f}; {r.ci_hi:.3f}]" if np.isfinite(r.ci_lo) else "NA", axis=1)
    tab8.to_csv(os.path.join(P["B_tab"], "tabel8_cross_dataset.csv"), index=False)
    pert_res[pert_res.dataset == "eksternal"].to_csv(os.path.join(P["B_tab"], "external_results.csv"), index=False)
    display(tab8.round(4))

    t8 = tab8[(tab8.model == "retrain") & tab8["Eksternal"].notna()]
    fig, ax = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1.6, 1]})
    x = np.arange(len(t8))
    ax[0].bar(x - 0.2, t8.Internal, 0.4, label="Internal", color="#2F6690")
    ax[0].bar(x + 0.2, t8.Eksternal, 0.4, label="External", color="#D1495B")
    ax[0].set_xticks(x); ax[0].set_xticklabels([s.replace(" (ROI referensi)", "\n(ref. ROI)").replace("Pipeline ", "Pipeline\n").replace("YOLO ", "YOLO\n")
                                                 .replace("Detection failure rate", "Failure\nrate") for s in t8.Metrik], fontsize=11)
    ax[0].set_ylabel("Value"); ax[0].legend(); ax[0].set_title("(a) Internal vs external (clean)")
    ax[1].errorbar(t8["Generalization gap"], np.arange(len(t8)), xerr=[t8["Generalization gap"] - t8.ci_lo, t8.ci_hi - t8["Generalization gap"]],
                   fmt="o", color="#333333", capsize=4)
    ax[1].axvline(0, color="gray", ls="--"); ax[1].set_yticks(np.arange(len(t8))); ax[1].set_yticklabels([LABEL_EN8.get(m, m) for m in t8.Metrik], fontsize=11)
    ax[1].invert_yaxis(); ax[1].set_xlabel("Generalization gap (95% CI)"); ax[1].set_title("(b) Generalization gap")
    dk.viz.show_and_save(fig, os.path.join(P["B_fig"], "Gambar8_internal_vs_eksternal"))
else:
    tab8 = pd.DataFrame()
    print("B7 dilewati: data eksternal tidak tersedia.")

In [ ]:
if ADA_EKSTERNAL and len(tab8):
    t = tab8[tab8.model == "retrain"].set_index("Metrik")
    teks = []
    for lab in t.index:
        r = t.loc[lab]
        if not np.isfinite(r.Eksternal):
            teks.append(f"{lab}: internal {fmt(r.Internal)}, eksternal NA ({r.catatan}).")
            continue
        teks.append(f"{lab}: internal {fmt(r.Internal)}, eksternal {fmt(r.Eksternal)}, gap {fmt(r['Generalization gap'])} "
                    f"(95% CI {fci(r.ci_lo, r.ci_hi)}" + ("; CI tidak mencakup nol)." if dk.teks.ci_excludes_zero(r.ci_lo, r.ci_hi) else "; CI mencakup nol)."))
    gd = t.loc["YOLO mAP@0.5"]; gc = t.loc["CNN macro-F1 (ROI referensi)"] if "CNN macro-F1 (ROI referensi)" in t.index else None
    det_turun = dk.teks.ci_excludes_zero(gd.ci_lo, gd.ci_hi) and gd["Generalization gap"] > 0
    if gc is not None and np.isfinite(gc.Eksternal):
        kls_turun = dk.teks.ci_excludes_zero(gc.ci_lo, gc.ci_hi) and gc["Generalization gap"] > 0
        if det_turun and kls_turun:
            teks.append("Deteksi dan klasifikasi sama-sama lebih rendah pada data eksternal; kontribusi kualitas ROI dan perubahan ciri visual perlu dipisahkan dengan analisis E1-E4 pada data eksternal.")
        elif (not det_turun) and kls_turun:
            teks.append("Deteksi tidak menunjukkan penurunan yang dapat dipastikan, sedangkan klasifikasi pada ROI referensi menurun; pola ini sejalan dengan pergeseran karakteristik visual ROI.")
        elif det_turun and not kls_turun:
            teks.append("Penurunan terutama terjadi pada lokalisasi YOLO, sedangkan klasifikasi ROI referensi tidak menunjukkan penurunan yang dapat dipastikan.")
        else:
            teks.append("Tidak ada penurunan deteksi maupun klasifikasi ROI referensi yang dapat dipastikan dari CI.")
    teks.append(f"Deskriptor domain: rerata L* internal {fmt(d_int.L_star.mean(), 2)} vs eksternal {fmt(d_ext.L_star.mean(), 2)}; "
                f"rerata a* internal {fmt(d_int.a_star.mean(), 2)} vs eksternal {fmt(d_ext.a_star.mean(), 2)}. "
                "Gap tidak ditafsirkan sebagai akibat satu faktor tanpa pengujian pendukung.")
    n_over = int(((ext["overlap_hash"] != "") | ext["overlap_sha"] | ext["overlap_source"]).sum())
    teks.append(f"Audit overlap mengeluarkan {n_over} citra eksternal yang identik/nyaris identik atau bersumber sama dengan data internal.")
    print("\n".join(teks))
LOG("B7 selesai.")

## B8. Analisis propagasi kesalahan (E1–E4)

### B8.1 Tabel 9 dan Tabel 10

Tabel 9 melaporkan *macro-F1* dan *balanced accuracy* E1–E4 per jenis dan level gangguan, ditambah agregat (rerata 18 kondisi, dihitung per replikasi *bootstrap*). Tabel 10 melaporkan ΔL, ΔC, ΔP, dan ΔLP beserta 95% CI berpasangan dan nilai p Holm untuk ΔLP. Jumlah lesi tanpa deteksi dan jumlah salah klasifikasi pada ROI valid dilaporkan terpisah.

In [ ]:
def propagasi(ms, ds, metrik="macro_f1"):
    d, est, boot = EVAL[(ms, ds)]
    key_ref, key_pred = f"cls_ref_{metrik}" if metrik == "macro_f1" else "cls_ref_bal_acc", f"cls_pred_{metrik}" if metrik == "macro_f1" else "cls_pred_bal_acc"
    E1, E2 = est["original"][key_ref], est["original"][key_pred]
    bE1, bE2 = boot["original"][key_ref].to_numpy(), boot["original"][key_pred].to_numpy()
    rows, bagg = [], {"E3": [], "E4": [], "dC": [], "dP": [], "dLP": []}
    for c in d["conds"]:
        if c == "original":
            continue
        info = next(x for x in CONDS if x["cond_id"] == c)
        E3, E4 = est[c][key_ref], est[c][key_pred]
        bE3, bE4 = boot[c][key_ref].to_numpy(), boot[c][key_pred].to_numpy()
        dL, dC, dP = bE1 - bE2, bE1 - bE3, bE2 - bE4
        dLP = (bE3 - bE4) - (bE1 - bE2)
        for k, v in [("E3", bE3), ("E4", bE4), ("dC", dC), ("dP", dP), ("dLP", dLP)]:
            bagg[k].append(v)
        roi = RAW[ms][ds][c][0]
        rows.append({"model": ms, "dataset": ds, "metrik": metrik, "cond_id": c, "ptype": info["ptype"], "level": info["level"], "value": info["value"],
                     "E1": E1, "E2": E2, "E3": E3, "E4": E4,
                     "E3_ci": dk.stats.percentile_ci(bE3), "E4_ci": dk.stats.percentile_ci(bE4),
                     "dL": E1 - E2, "dL_ci": dk.stats.percentile_ci(dL), "dC": E1 - E3, "dC_ci": dk.stats.percentile_ci(dC),
                     "dP": E2 - E4, "dP_ci": dk.stats.percentile_ci(dP), "dLP": (E3 - E4) - (E1 - E2), "dLP_ci": dk.stats.percentile_ci(dLP),
                     "p_dC": dk.stats.boot_p_two_sided(dC), "p_dP": dk.stats.boot_p_two_sided(dP), "p_dLP": dk.stats.boot_p_two_sided(dLP),
                     "n_lesi": len(roi), "n_tanpa_roi_valid": int((roi.pipe_pred == -1).sum()),
                     "n_tanpa_deteksi": int((~roi.has_overlap.astype(bool)).sum()),
                     "n_salah_pada_roi_valid": int(((roi.pipe_pred >= 0) & (roi.pipe_pred != roi.true)).sum())})
    df = pd.DataFrame(rows)
    for k in ["p_dC", "p_dP", "p_dLP"]:
        df[k.replace("p_", "p_holm_")] = dk.stats.holm(df[k].to_numpy())
    agg = {"cond_id": "AGREGAT (rerata 18 kondisi)", "model": ms, "dataset": ds, "metrik": metrik, "E1": E1, "E2": E2,
           "E3": df.E3.mean(), "E4": df.E4.mean(), "dL": E1 - E2, "dC": df.dC.mean(), "dP": df.dP.mean(), "dLP": df.dLP.mean()}
    for k, src in [("E3_ci", "E3"), ("E4_ci", "E4"), ("dC_ci", "dC"), ("dP_ci", "dP"), ("dLP_ci", "dLP")]:
        agg[k] = dk.stats.percentile_ci(np.nanmean(np.vstack(bagg[src]), axis=0))
    agg["dL_ci"] = dk.stats.percentile_ci(bE1 - bE2)
    agg["E1_ci"], agg["E2_ci"] = dk.stats.percentile_ci(bE1), dk.stats.percentile_ci(bE2)
    df["E1_ci"], df["E2_ci"] = [agg["E1_ci"]] * len(df), [agg["E2_ci"]] * len(df)
    return pd.concat([df, pd.DataFrame([agg])], ignore_index=True)

EP = pd.concat([propagasi(ms, ds, m) for (ms, ds) in EVAL for m in ["macro_f1", "bal_acc"]
                if not (ds == "eksternal" and not EXT_ADA_KEPARAHAN) and len(EVAL[(ms, ds)][0]["conds"]) > 1], ignore_index=True)
for col in [c for c in EP.columns if c.endswith("_ci")]:
    EP[col + "_lo"] = EP[col].map(lambda t: t[0] if isinstance(t, tuple) else np.nan)
    EP[col + "_hi"] = EP[col].map(lambda t: t[1] if isinstance(t, tuple) else np.nan)
EP_out = EP.drop(columns=[c for c in EP.columns if c.endswith("_ci")])
EP_out["label_map_status"] = STATUS_LABEL_MAP
EP_out.to_csv(os.path.join(P["B_tab"], "error_propagation.csv"), index=False)

def ci_str(v, lo, hi):
    return f"{v:.3f} [{lo:.3f}; {hi:.3f}]" if np.isfinite(v) else "NA"
s = EP[(EP.model == "retrain") & (EP.dataset == "internal") & (EP.metrik == "macro_f1")]
tab9 = pd.DataFrame({"Kondisi gangguan": s.cond_id,
                     "E1 (asli, ROI ref.)": [ci_str(a, *b) for a, b in zip(s.E1, s.E1_ci)],
                     "E2 (asli, ROI YOLO)": [ci_str(a, *b) for a, b in zip(s.E2, s.E2_ci)],
                     "E3 (terganggu, ROI ref.)": [ci_str(a, *b) for a, b in zip(s.E3, s.E3_ci)],
                     "E4 (terganggu, ROI YOLO)": [ci_str(a, *b) for a, b in zip(s.E4, s.E4_ci)]})
sb = EP[(EP.model == "retrain") & (EP.dataset == "internal") & (EP.metrik == "bal_acc")]
tab9["Balanced acc. E3"] = [f"{v:.3f}" for v in sb.E3]; tab9["Balanced acc. E4"] = [f"{v:.3f}" for v in sb.E4]
tab9.to_csv(os.path.join(P["B_tab"], "tabel9_E1_E4.csv"), index=False)
tab10 = pd.DataFrame({"Kondisi gangguan": s.cond_id,
                      "dL": [ci_str(a, *b) for a, b in zip(s.dL, s.dL_ci)], "dC": [ci_str(a, *b) for a, b in zip(s.dC, s.dC_ci)],
                      "dP": [ci_str(a, *b) for a, b in zip(s.dP, s.dP_ci)], "dLP (95% CI)": [ci_str(a, *b) for a, b in zip(s.dLP, s.dLP_ci)],
                      "p Holm dLP": s.p_holm_dLP.round(4), "Lesi tanpa ROI valid": s.n_tanpa_roi_valid,
                      "Salah pada ROI valid": s.n_salah_pada_roi_valid})
tab10.to_csv(os.path.join(P["B_tab"], "tabel10_propagasi_kesalahan.csv"), index=False)
display(tab9); display(tab10)

### B8.2 Gambar 9: penalti lokalisasi, degradasi klasifikasi, dan interaksi

Setiap panel adalah satu jenis gangguan. Batang menunjukkan ΔC, ΔP, dan ΔLP per level dengan 95% CI; garis putus-putus adalah ΔL pada citra asli (konstan untuk semua gangguan).

In [ ]:
s = EP[(EP.model == "retrain") & (EP.dataset == "internal") & (EP.metrik == "macro_f1") & (EP.cond_id.str.contains("_L"))]
ptypes = [p for p in dk.viz.PTYPE_ORDER if p in SPEC]
fig, ax = plt.subplots(1, len(ptypes), figsize=(3.3 * len(ptypes), 4.6), sharey=True)
for a, pt in zip(np.atleast_1d(ax), ptypes):
    q = s[s.ptype == pt].sort_values("level")
    x = np.arange(len(q))
    for k, (col, lab, c) in enumerate([("dC", "ΔC", "#2F6690"), ("dP", "ΔP", "#D1495B"), ("dLP", "ΔLP", "#4C9F70")]):
        lo = np.array([t[0] for t in q[col + "_ci"]]); hi = np.array([t[1] for t in q[col + "_ci"]])
        a.bar(x + (k - 1) * 0.27, q[col], 0.27, yerr=[q[col] - lo, hi - q[col]], capsize=3, color=c, label=lab)
    a.axhline(s.dL.iloc[0], color="black", ls="--", lw=1.2, label="ΔL (clean)")
    a.axhline(0, color="gray", lw=0.8)
    a.set_xticks(x); a.set_xticklabels([f"L{int(l)}\n{float(v):g}" for l, v in zip(q.level, q.value)])
    a.set_title(SPEC[pt].get("label_en", pt))
np.atleast_1d(ax)[0].set_ylabel("Δ macro-F1")
h, l = np.atleast_1d(ax)[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.12), frameon=False)
dk.viz.show_and_save(fig, os.path.join(P["B_fig"], "Gambar9_propagasi_kesalahan"))

r = EP[(EP.model == "retrain") & (EP.dataset == "internal") & (EP.metrik == "macro_f1")]
agg = r[r.cond_id.str.startswith("AGREGAT")].iloc[0]; per = r[~r.cond_id.str.startswith("AGREGAT")]
teks = [f"Pada citra asli, E1 = {fmt(agg.E1)} dan E2 = {fmt(agg.E2)}; Delta L = {fmt(agg.dL)} (95% CI {fci(*agg.dL_ci)}), "
        + ("penalti ROI prediksi dapat dipastikan dari CI." if dk.teks.ci_excludes_zero(*agg.dL_ci) else "CI mencakup nol.")]
w = per.loc[per.dLP.abs().idxmax()]
teks.append(f"Perubahan penalti lokalisasi terbesar (|Delta LP|) terjadi pada {w.cond_id} (nilai {w.value}): Delta LP = {fmt(w.dLP)} "
            f"(95% CI {fci(*w.dLP_ci)}, p Holm = {fmt(w.p_holm_dLP, 4)}).")
pos = per[(per.p_holm_dLP < 0.05) & (per.dLP > 0)]; neg = per[(per.p_holm_dLP < 0.05) & (per.dLP < 0)]
teks.append(f"Delta LP positif dengan p Holm < 0,05 (penalti ROI prediksi membesar saat citra terganggu): {len(pos)} kondisi"
            + (f" ({', '.join(pos.cond_id)})" if len(pos) else "") + f"; negatif: {len(neg)} kondisi" + (f" ({', '.join(neg.cond_id)})" if len(neg) else "") + ".")
teks.append(f"Agregat 18 kondisi: Delta C = {fmt(agg.dC)} (95% CI {fci(*agg.dC_ci)}), Delta P = {fmt(agg.dP)} (95% CI {fci(*agg.dP_ci)}), "
            f"Delta LP = {fmt(agg.dLP)} (95% CI {fci(*agg.dLP_ci)}).")
dom = per.assign(rasio=per.dC / per.dP.replace(0, np.nan))
teks.append("Kondisi dengan Delta C mendekati Delta P menandakan degradasi pipeline didominasi degradasi klasifikasi; "
            f"median rasio Delta C / Delta P = {fmt(dom.rasio.median(), 2)} (hanya deskriptif karena macro-F1 nonlinier).")
print("\n".join(teks))

### B8.3 Hubungan kualitas lokalisasi dengan kesalahan klasifikasi (Tabel 11, Gambar 10)

Setiap lesi referensi dipasangkan dengan satu kotak prediksi (Hungarian, IoU > 0) pada `CONF_THR`. CNN dijalankan pada kotak tersebut walaupun IoU-nya rendah, sehingga hubungan IoU dengan kesalahan dapat diamati pada seluruh rentang. Lesi tanpa kotak yang tumpang tindih dikelompokkan terpisah dan tidak diberi nilai IoU. Untuk setiap rentang dihitung jumlah lesi, *classification error rate*, dan rerata probabilitas kelas benar, dengan 95% CI *cluster bootstrap*.

Kekuatan hubungan diuji dengan regresi logistik GEE (*Generalized Estimating Equations*, Liang & Zeger, 1986) `error ~ IoU` dengan klaster = kelompok anti-*leakage*, struktur korelasi kerja independen, dan galat baku *robust (sandwich)* yang tetap sahih untuk data berklaster; *odds ratio* dilaporkan per kenaikan IoU 0,1.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
def tabel_iou(roi, B=B_BOOT, seed=SEED):
    r = roi.copy()
    r["bin"] = pd.cut(r["iou"].where(r.has_overlap.astype(bool)), IOU_BINS, right=False,
                      labels=["IoU < 0,25", "0,25 <= IoU < 0,50", "0,50 <= IoU < 0,75", "IoU >= 0,75"])
    r["err"] = (r.any_pred != r.true).astype(float).where(r.has_overlap.astype(bool))
    r["p_true"] = r[["any_p0", "any_p1", "any_p2"]].to_numpy()[np.arange(len(r)), r.true.to_numpy()]
    labs = list(r["bin"].cat.categories)
    est = {b: (int((r.bin == b).sum()), r.loc[r.bin == b, "err"].mean(), r.loc[r.bin == b, "p_true"].mean()) for b in labs}
    cb = dk.stats.ClusterBootstrap(B, seed).register("r", r.group.astype(str).values)
    bs = {b: [] for b in labs}
    for dr in cb.draws():
        rr = r.iloc[dr["r"]]
        for b in labs:
            m = rr.bin == b
            bs[b].append((rr.loc[m, "err"].mean(), rr.loc[m, "p_true"].mean()))
    rows = []
    for b in labs:
        arr = np.array(bs[b], dtype=float)
        rows.append({"Rentang IoU": b, "Jumlah lesi": est[b][0], "Classification error rate": est[b][1],
                     "err_ci_lo": dk.stats.percentile_ci(arr[:, 0])[0], "err_ci_hi": dk.stats.percentile_ci(arr[:, 0])[1],
                     "Mean confidence of true class": est[b][2],
                     "p_ci_lo": dk.stats.percentile_ci(arr[:, 1])[0], "p_ci_hi": dk.stats.percentile_ci(arr[:, 1])[1]})
    nd = int((~r.has_overlap.astype(bool)).sum())
    rows.append({"Rentang IoU": "Tanpa deteksi", "Jumlah lesi": nd, "Classification error rate": np.nan,
                 "Mean confidence of true class": np.nan, "catatan": "gagal end-to-end"})
    g = r[r.has_overlap.astype(bool)].copy()
    g["iou10"] = g["iou"] * 10
    gee = {}
    try:
        fit = smf.gee("err ~ iou10", groups=g["group"].astype(str), data=g, family=sm.families.Binomial(),
                      cov_struct=sm.cov_struct.Independence()).fit()
        ci = fit.conf_int().loc["iou10"]
        gee = {"OR_per_0.1_IoU": float(np.exp(fit.params["iou10"])), "OR_ci_lo": float(np.exp(ci[0])), "OR_ci_hi": float(np.exp(ci[1])),
               "p": float(fit.pvalues["iou10"]), "n": len(g), "n_klaster": g.group.nunique()}
    except Exception as e:
        gee = {"error": str(e)[:200]}
    return pd.DataFrame(rows), gee, r

roi_clean = RAW["retrain"]["internal"]["original"][0]
tab11, gee11, r11 = tabel_iou(roi_clean)
tab11.to_csv(os.path.join(P["B_tab"], "tabel11_iou_vs_error.csv"), index=False)
pool = pd.concat([RAW["retrain"]["internal"][c][0] for c in RAW["retrain"]["internal"] if c != "original"], ignore_index=True)
tab11b, gee11b, _ = tabel_iou(pool, B=min(B_BOOT, 300))
tab11b.to_csv(os.path.join(P["B_tab"], "tabel11b_iou_vs_error_semua_perturbasi.csv"), index=False)
dk.core.save_json({"clean": gee11, "perturbasi_gabungan": gee11b}, os.path.join(P["B_tab"], "gee_iou_error.json"))
display(tab11.round(4)); print("GEE (clean):", gee11); print("GEE (gabungan 18 kondisi):", gee11b)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
for a, (t, judul) in zip(ax, [(tab11, "(a) Clean test images"), (tab11b, "(b) Pooled perturbed conditions")]):
    q = t[t["Rentang IoU"] != "Tanpa deteksi"]
    x = np.arange(len(q))
    a.bar(x, q["Classification error rate"], yerr=[q["Classification error rate"] - q.err_ci_lo, q.err_ci_hi - q["Classification error rate"]],
          capsize=4, color="#D1495B", label="Error rate")
    a2 = a.twinx(); a2.plot(x, q["Mean confidence of true class"], "o-", color="#2F6690", label="Mean p(true class)")
    a2.set_ylim(0, 1); a2.set_ylabel("Mean p(true class)"); a2.grid(False)
    a.set_xticks(x); a.set_xticklabels(["< 0.25", "0.25–0.50", "0.50–0.75", "≥ 0.75"][:len(q)])
    a.set_xlabel("IoU of matched YOLO box"); a.set_ylabel("Classification error rate"); a.set_ylim(0, 1)
    for xi, n in zip(x, q["Jumlah lesi"]):
        a.text(xi, 0.02, f"n={n:,}", ha="center", fontsize=11, color="white" if n else "black")
    nd = int(t.loc[t["Rentang IoU"] == "Tanpa deteksi", "Jumlah lesi"].iloc[0])
    a.set_title(f"{judul}\n(no detection: n={nd:,})")
dk.viz.show_and_save(fig, os.path.join(P["B_fig"], "Gambar10_IoU_vs_error"))

q = tab11[tab11["Rentang IoU"] != "Tanpa deteksi"]
teks = ["Rentang IoU: " + "; ".join(f"{a}: n = {n:,}, error {fmt(e)}, rerata p(kelas benar) {fmt(p)}" for a, n, e, p in
                                    zip(q["Rentang IoU"], q["Jumlah lesi"], q["Classification error rate"], q["Mean confidence of true class"]))]
teks.append(f"Lesi tanpa deteksi: {int(tab11.loc[tab11['Rentang IoU'] == 'Tanpa deteksi', 'Jumlah lesi'].iloc[0]):,} (dihitung gagal end-to-end).")
if "OR_per_0.1_IoU" in gee11:
    teks.append(f"GEE: odds ratio kesalahan per kenaikan IoU 0,1 = {fmt(gee11['OR_per_0.1_IoU'])} (95% CI {fci(gee11['OR_ci_lo'], gee11['OR_ci_hi'])}, "
                f"p = {fmt(gee11['p'], 4)}; {gee11['n']:,} lesi dalam {gee11['n_klaster']} klaster). "
                + ("Hubungan negatif IoU dengan kesalahan didukung uji ini." if gee11["OR_ci_hi"] < 1 else
                   "Hubungan positif IoU dengan kesalahan teramati." if gee11["OR_ci_lo"] > 1 else "CI odds ratio mencakup 1, sehingga arah hubungan belum dapat dipastikan."))
if gee11.get("n_klaster", 99) < 40:
    teks.append(f"Jumlah klaster GEE hanya {gee11.get('n_klaster')}; galat baku sandwich cenderung terlalu kecil bila klaster < 40 (Mancl & DeRouen, 2001), sehingga nilai p ditafsirkan hati-hati.")
teks.append("IoU mengukur kesesuaian spasial dan tidak mengukur langsung kelengkapan ciri klinis di dalam ROI.")
print("\n".join(teks))
LOG("B8 selesai.")

## B9. Statistik dan reliabilitas prediksi

### B9.1 Kalibrasi, *Brier score*, dan metrik per kelas (Tabel 12, `statistics.csv`)

Kalibrasi dievaluasi pada probabilitas *softmax* CNN tanpa kalibrasi tambahan:

- **ROI referensi** (E1/E3): seluruh lesi referensi;
- ***Pipeline*** (E2/E4): lesi dengan ROI prediksi valid. Lesi tanpa ROI valid tidak memiliki probabilitas dan dilaporkan sebagai *failure rate*.

Kondisi *Perturbed* didefinisikan secara transparan sebagai gabungan 18 kondisi (setiap lesi muncul 18 kali; klaster *bootstrap* tetap kelompok sumber). Nilai per kondisi tersedia di `statistics.csv`. *Multiclass Brier score* berada pada rentang 0–2 (lebih kecil lebih baik).

In [ ]:
def kalibrasi(rois, sumber, B=B_BOOT, seed=SEED):
    r = pd.concat(rois, ignore_index=True)
    if sumber == "ref":
        probs, y, g = r[["ref_p0", "ref_p1", "ref_p2"]].to_numpy(float), r.true.to_numpy(int), r.group.astype(str).to_numpy()
        pred = r.ref_pred.to_numpy(int)
    else:
        m = r.valid_roi.astype(bool).to_numpy()
        probs, y, g = r.loc[m, ["any_p0", "any_p1", "any_p2"]].to_numpy(float), r.true.to_numpy(int)[m], r.group.astype(str).to_numpy()[m]
        pred = r.pipe_pred.to_numpy(int)
    yall, gall = r.true.to_numpy(int), r.group.astype(str).to_numpy()
    est = {"macro_f1": dk.metrics.macro_f1_fast(yall, pred), "ece": dk.metrics.ece_top(probs, y, N_BIN_ECE),
           "classwise_ece": dk.metrics.classwise_ece(probs, y, N_BIN_ECE), "brier": dk.metrics.brier_multiclass(probs, y),
           "n_lesi": len(yall), "n_prob": len(y)}
    cm = dk.metrics.cls_metrics(yall, pred)
    est.update({k: v for k, v in cm.items() if k.startswith(("precision_", "recall_", "f1_", "support_"))})
    cb = dk.stats.ClusterBootstrap(B, seed).register("all", gall).register("prob", g)
    bs = []
    for dr in cb.draws():
        a, p = dr["all"], dr["prob"]
        cmb = dk.metrics.cls_metrics(yall[a], pred[a])
        bs.append({"macro_f1": cmb["macro_f1"], "ece": dk.metrics.ece_top(probs[p], y[p], N_BIN_ECE),
                   "classwise_ece": dk.metrics.classwise_ece(probs[p], y[p], N_BIN_ECE), "brier": dk.metrics.brier_multiclass(probs[p], y[p]),
                   **{f"f1_{k}": cmb.get(f"f1_{k}", np.nan) for k in range(3)}})
    bs = pd.DataFrame(bs)
    for k in ["macro_f1", "ece", "classwise_ece", "brier", "f1_0", "f1_1", "f1_2"]:
        est[k + "_ci_lo"], est[k + "_ci_hi"] = dk.stats.percentile_ci(bs[k])
    return est, probs, y

STAT, REL = [], {}
for (ms, ds) in EVAL:
    if ds == "eksternal" and not EXT_ADA_KEPARAHAN:
        continue
    raw = RAW[ms][ds]
    kondisi = {"Clean": ["original"], "Perturbed (gabungan 18 kondisi)": [c for c in raw if c != "original"]}
    kondisi.update({c: [c] for c in raw if c != "original"})
    for kname, cl in kondisi.items():
        if not cl:
            continue
        for sumber in ["ref", "pipeline"]:
            Bk = B_BOOT if kname == "Clean" else min(B_BOOT, 300) if kname.startswith("Perturbed") else min(B_BOOT, 200)
            e, pr, yy = kalibrasi([raw[c][0] for c in cl], sumber, B=Bk)
            STAT.append({"model": ms, "dataset": ds, "kondisi": kname, "sumber_roi": sumber, **e})
            if kname in ("Clean", "Perturbed (gabungan 18 kondisi)"):
                REL[(ms, ds, kname, sumber)] = (pr, yy)
stat_df = pd.DataFrame(STAT)
stat_df["label_map_status"] = STATUS_LABEL_MAP
stat_df.to_csv(os.path.join(P["B_tab"], "statistics.csv"), index=False)

rows = []
for ds, dsl in [("internal", "Internal"), ("eksternal", "External")]:
    for kname, kl in [("Clean", "Clean"), ("Perturbed (gabungan 18 kondisi)", "Perturbed")]:
        for sumber, sl in [("ref", "ROI referensi"), ("pipeline", "Pipeline (ROI YOLO)")]:
            q = stat_df[(stat_df.model == "retrain") & (stat_df.dataset == ds) & (stat_df.kondisi == kname) & (stat_df.sumber_roi == sumber)]
            if q.empty:
                rows.append({"Kondisi": f"{dsl} - {kl}", "Sumber ROI": sl, "Macro-F1 (95% CI)": "NA", "ECE (95% CI)": "NA", "Brier score (95% CI)": "NA"})
                continue
            q = q.iloc[0]
            rows.append({"Kondisi": f"{dsl} - {kl}", "Sumber ROI": sl,
                         "Macro-F1 (95% CI)": ci_str(q.macro_f1, q.macro_f1_ci_lo, q.macro_f1_ci_hi),
                         "ECE (95% CI)": ci_str(q.ece, q.ece_ci_lo, q.ece_ci_hi),
                         "Brier score (95% CI)": ci_str(q.brier, q.brier_ci_lo, q.brier_ci_hi), "n lesi": int(q.n_lesi), "n dengan probabilitas": int(q.n_prob)})
tab12 = pd.DataFrame(rows)
tab12.to_csv(os.path.join(P["B_tab"], "tabel12_statistik_kalibrasi.csv"), index=False)
display(tab12)

### B9.2 *Reliability diagram* (Gambar 11)

Titik adalah akurasi empiris per bin *confidence* (15 bin lebar sama); ukuran titik sebanding jumlah lesi dalam bin. Garis diagonal adalah kalibrasi sempurna. Panel memakai probabilitas CNN pada ROI referensi agar perbedaan antar-kondisi tidak tercampur dengan kegagalan deteksi.

In [ ]:
panels = [("internal", "Clean", "Internal - clean"), ("internal", "Perturbed (gabungan 18 kondisi)", "Internal - perturbed"),
          ("eksternal", "Clean", "External - clean"), ("eksternal", "Perturbed (gabungan 18 kondisi)", "External - perturbed")]
panels = [p for p in panels if ("retrain", p[0], p[1], "ref") in REL]
fig, ax = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 4.8), squeeze=False)
for a, (ds, kname, judul) in zip(ax[0], panels):
    pr, yy = REL[("retrain", ds, kname, "ref")]
    rt = pd.DataFrame(dk.metrics.reliability_table(pr, yy, N_BIN_ECE)).dropna()
    a.plot([0, 1], [0, 1], "--", color="gray")
    a.scatter(rt.conf_mean, rt.acc_mean, s=20 + 400 * rt.n / rt.n.max(), color="#2F6690", alpha=0.8, zorder=3)
    a.plot(rt.conf_mean, rt.acc_mean, color="#2F6690")
    a.set_xlim(0, 1); a.set_ylim(0, 1); a.set_xlabel("Confidence"); a.set_ylabel("Accuracy")
    a.set_title(f"{judul}\nECE = {dk.metrics.ece_top(pr, yy, N_BIN_ECE):.3f}")
dk.viz.show_and_save(fig, os.path.join(P["B_fig"], "Gambar11_reliability_diagram"))

q = stat_df[(stat_df.model == "retrain") & (stat_df.sumber_roi == "ref") & stat_df.kondisi.isin(["Clean", "Perturbed (gabungan 18 kondisi)"])]
teks = []
for r in q.itertuples():
    teks.append(f"{r.dataset} / {r.kondisi} (ROI referensi): macro-F1 {fmt(r.macro_f1)} (95% CI {fci(r.macro_f1_ci_lo, r.macro_f1_ci_hi)}), "
                f"ECE {fmt(r.ece)} (95% CI {fci(r.ece_ci_lo, r.ece_ci_hi)}), Brier {fmt(r.brier)} (95% CI {fci(r.brier_ci_lo, r.brier_ci_hi)}).")
c = stat_df[(stat_df.model == "retrain") & (stat_df.dataset == "internal") & (stat_df.sumber_roi == "ref") & (stat_df.kondisi == "Clean")].iloc[0]
teks.append("F1 per kelas (internal clean, ROI referensi): " + ", ".join(
    f"{NAMA_KELAS_ID[k]} {fmt(getattr(c, f'f1_{k}'))} (95% CI {fci(getattr(c, f'f1_{k}_ci_lo'), getattr(c, f'f1_{k}_ci_hi'))})" for k in range(3)) + ".")
per = stat_df[(stat_df.model == "retrain") & (stat_df.dataset == "internal") & (stat_df.sumber_roi == "ref") & stat_df.kondisi.str.contains("_L")]
if len(per):
    w = per.loc[per.ece.idxmax()]
    teks.append(f"ECE tertinggi di antara 18 kondisi internal: {w.kondisi} ({fmt(w.ece)}). Rerata confidence yang melebihi akurasi pada bin tinggi menandakan overconfidence; "
                "arah ini dibaca langsung dari Gambar 11.")
print("\n".join(teks))
LOG("B9 selesai.")

## B10. Ekspor luaran artikel 2025

Sel berikut:

1. menggabungkan Tabel 4–12 dan tabel pendukung ke satu *workbook* `PFR_2025_Tabel_Hasil.xlsx` (satu *sheet* per tabel, ditambah *sheet* README berisi status label map, status data eksternal, dan catatan *in-sample* bobot lama);
2. membuat montase contoh kegagalan (lesi tidak terdeteksi dan salah klasifikasi pada ROI valid);
3. menulis `README_hasil.md`, menampilkan *checklist* B6–B10, lalu mengemas folder `PFR_2025_ROBUSTNESS` (tanpa `raw/`) menjadi ZIP dan mengunduhnya.

In [ ]:
roi = RAW["retrain"]["internal"]["original"][0]
by_sid = {i["sample_id"]: i for i in items_test}
miss = roi[~roi.has_overlap.astype(bool)].drop_duplicates("sample_id").head(4)
wrong = roi[(roi.pipe_pred >= 0) & (roi.pipe_pred != roi.true)].sort_values("det_conf", ascending=False).drop_duplicates("sample_id").head(4)
fig, ax = plt.subplots(2, 4, figsize=(16, 8.4))
yr, cr = MODEL_SETS["retrain"]
for k in range(4):
    for row_i, (df_, judul) in enumerate([(miss, "Missed lesion"), (wrong, "Misclassified (valid ROI)")]):
        a = ax[row_i, k]; a.axis("off")
        if k >= len(df_):
            continue
        r = df_.iloc[k]; it = by_sid[r.sample_id]
        im = dk.data.read_rgb(it["path"])
        pr, _ = yr.predict([im], conf=CONF_THR)
        a.imshow(im)
        for b in pr[0]:
            a.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="#2F6690", lw=1.2))
        g = it["gt_xyxy"][int(r.roi_idx)]
        a.add_patch(plt.Rectangle((g[0], g[1]), g[2] - g[0], g[3] - g[1], fill=False, ec="#D1495B", lw=2.4))
        sub = f"true {NAMA_KELAS[int(r.true)]}" + (f", pred {NAMA_KELAS[int(r.pipe_pred)]}" if r.pipe_pred >= 0 else ", no ROI")
        a.set_title(f"{judul}\n{sub}", fontsize=12)
dk.viz.show_and_save(fig, os.path.join(P["B_fig"], "GambarS1_contoh_kegagalan"))
print("Kotak merah = lesi referensi yang dibahas; kotak biru = deteksi YOLO pada CONF_THR.")

In [ ]:
xlsx = os.path.join(P["B"], "PFR_2025_Tabel_Hasil.xlsx")
tab5 = pd.read_csv(os.path.join(P["A_tab"], "tabel5_baseline.csv"))
readme = pd.DataFrame({"butir": ["label_map_status", "split_sha256", "model utama", "bobot lama", "data eksternal", "B_BOOT", "unit bootstrap",
                                 "definisi E3/E4", "dibuat"],
                       "nilai": [STATUS_LABEL_MAP, lock["sha256_split"], "retrain (YOLO satu kelas + EfficientNet-B0)",
                                 "in-sample terhadap data internal" if "lama" in MODEL_SETS else "tidak tersedia",
                                 json.dumps(ext_meta) if ADA_EKSTERNAL else "tidak tersedia (klaim terbatas pada robustness internal)",
                                 B_BOOT, "kelompok sumber/sesi/near-duplicate", "E3 = terganggu + ROI referensi; E4 = terganggu + ROI YOLO",
                                 datetime.datetime.now().isoformat(timespec="seconds")]})
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    readme.to_excel(w, sheet_name="README", index=False)
    for nm, df_ in [("Tabel4_Dataset", tab4), ("Tabel5_Baseline", tab5), ("Tabel6_Deteksi", tab6), ("Tabel7_Klasifikasi", tab7),
                    ("Tabel8_CrossDataset", tab8), ("Tabel9_E1_E4", tab9), ("Tabel10_Propagasi", tab10), ("Tabel11_IoU", tab11),
                    ("Tabel11b_IoU_Perturbasi", tab11b), ("Tabel12_Kalibrasi", tab12), ("Statistik_lengkap", stat_df),
                    ("Hasil_perturbasi_long", pert_res)]:
        if isinstance(df_, pd.DataFrame) and len(df_):
            df_.to_excel(w, sheet_name=nm[:31], index=False)
    if ADA_EKSTERNAL:
        ext.drop(columns=["phash", "dhash", "label_ids"]).to_excel(w, sheet_name="Audit_overlap_eksternal", index=False)
print("Workbook:", xlsx)

with open(os.path.join(P["B"], "README_hasil.md"), "w", encoding="utf-8") as f:
    f.write("# PFR_2025_ROBUSTNESS\n\n")
    f.write(f"- Status label map: {STATUS_LABEL_MAP}\n- SHA-256 split: {lock['sha256_split']}\n")
    f.write(f"- Data eksternal: {'tersedia ' + json.dumps(ext_meta) if ADA_EKSTERNAL else 'tidak tersedia'}\n")
    f.write("- Bobot lama: in-sample terhadap data internal (tidak dipakai sebagai bukti generalisasi internal)\n\n## Berkas\n\n")
    for root, _, files in os.walk(P["B"]):
        if "/raw" in root:
            continue
        for fn in sorted(files):
            f.write(f"- {os.path.relpath(os.path.join(root, fn), P['B'])}\n")

cek = {"06 Uji 6 perturbasi": ["tables/perturbation_results.csv", "figures/Gambar6_robustness_deteksi_YOLO.png"],
       "07 Dataset eksternal independen": ["tables/external_results.csv"],
       "08 Analisis propagasi kesalahan": ["tables/error_propagation.csv"],
       "09 Kalibrasi dan interval kepercayaan": ["tables/statistics.csv", "figures/Gambar11_reliability_diagram.png"],
       "10 Ekspor luaran artikel 2025": ["PFR_2025_Tabel_Hasil.xlsx"]}
for k, fs in cek.items():
    print(("[x] " if all(os.path.exists(os.path.join(P["B"], x)) for x in fs) else "[ ] ") + k)

tmp = os.path.join(WORK, "_zip_B")
if os.path.exists(tmp):
    shutil.rmtree(tmp)
shutil.copytree(P["B"], os.path.join(tmp, "PFR_2025_ROBUSTNESS"), ignore=shutil.ignore_patterns("raw"))
zp = shutil.make_archive(os.path.join(WORK, f"{NAMA_PROYEK}_PFR_2025_ROBUSTNESS"), "zip", tmp)
shutil.copy2(zp, ROOT)
print("ZIP:", zp, f"({os.path.getsize(zp) / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(zp)
LOG("B10 selesai.")